In [136]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP
from dataclasses import dataclass
from datetime import date
from abc import ABC, abstractmethod
from hypothesis import given, strategies as st

In [137]:
class Money:
    def __init__(self, amount):
        self.amount = Decimal(str(amount)).quantize(Decimal("0.01"),rounding=ROUND_HALF_UP)
    def __add__(self, other):
        return Money(self.amount + other.amount)
    def __sub__(self, other):
        return Money(self.amount - other.amount)
    def __mul__(self, other):
        return Money(self.amount * other)
    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount < other.amount
    def __repr__(self):
        return f"Money({self.amount})"
    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount
    def __ge__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount >= other.amount


In [138]:
@dataclass
class TaxContext:
    gross_income: Money
    allowances: Money
    residency_status: str
    effective_date: date

In [139]:
@dataclass
class TaxResult:
    amount: Money

In [140]:
class Tax(ABC):
    @abstractmethod
    def calculate(self, ctx: TaxContext) -> TaxResult:
        pass

In [141]:
class TaxError(Exception):
    pass

class InvalidIncomeError(TaxError):
    pass

class NoScheduleForDateError(TaxError):
    pass

In [142]:
class PAYE(Tax):
    def calculate(self, ctx: TaxContext) -> TaxResult:
            income = ctx.gross_income.amount + ctx.allowances.amount
            if ctx.effective_date < date(2026, 7, 1):
                raise NoScheduleForDateError(
                    "This PAYE schedule applies from 1 July 2026"
                )

            if ctx.residency_status == "resident":
             return self._calculate_resident(income)

            elif ctx.residency_status == "non-resident":
                return self._calculate_nonresident(income)

            else:
                raise ValueError(
                "residency_status must be 'resident' or 'non-resident'"
            )
    def _calculate_resident(self, income: Decimal) -> TaxResult:
            if income < 0:
                raise InvalidIncomeError("Income cannot be negative")

            if income <= Decimal("335000"):
                tax = Money("0")

            elif income <= Decimal("410000"):
                taxable_amount = income - Decimal("335000")
                tax = Money(taxable_amount * Decimal("0.20"))
            elif income <= Decimal("485000"):
                taxable_amount = income - Decimal("410000")
                tax = Money(
                Decimal("15000") + taxable_amount * Decimal("0.25"))
            elif income <= Decimal("10000000"):
                taxable_amount = income - Decimal("485000")
                tax = Money(
                Decimal("33750") + taxable_amount * Decimal("0.30"))

            else:
                taxable_amount = income - Decimal("10000000")
                tax = Money(
                Decimal("2888250") + taxable_amount * Decimal("0.10"))
                
            return TaxResult(amount=tax)
    def _calculate_nonresident(self, income: Decimal) -> TaxResult:
        if income < 0:
            raise InvalidIncomeError("Income cannot be negative")

        if income <= Decimal("335000"):
            tax = Money(income * Decimal("0.10"))

        elif income <= Decimal("410000"):
            taxable_amount = income - Decimal("335000")
            tax = Money(
                Decimal("33500") + taxable_amount * Decimal("0.20")
            )

        elif income <= Decimal("10000000"):
            taxable_amount = income - Decimal("410000")
            tax = Money(
                Decimal("48500") + taxable_amount * Decimal("0.30")
            )

        else:
            taxable_amount = income - Decimal("10000000")
            tax = Money(
                Decimal("2925500") + taxable_amount * Decimal("0.10")
            )

        return TaxResult(amount=tax)


In [143]:
class NSSF:
    def __init__(self):
        self.employee_rate = Decimal("0.05")
        self.employer_rate = Decimal("0.10")
    def calculate(self, gross_income: Money) -> dict:
        if gross_income.amount < 0:
            raise InvalidIncomeError("Income cannot be negative")

        employee_contribution = Money(
            gross_income.amount * self.employee_rate
        )

        employer_contribution = Money(
            gross_income.amount * self.employer_rate
        )

        return {
            "employee": employee_contribution,
            "employer": employer_contribution
        }

In [144]:
class LST:
    def __init__(self, amount=0):
        self.amount = Money(amount)

    def calculate(self) -> Money:
        return self.amount

In [146]:

class Payroll:
    def __init__(self, paye, nssf, lst):
        self.paye = paye
        self.nssf = nssf
        self.lst = lst

    def calculate(self, ctx: TaxContext) -> dict:
        nssf_result = self.nssf.calculate(ctx.gross_income)
        lst_amount = self.lst.calculate()
        paye_result = self.paye.calculate(ctx)

        total_earnings = ctx.gross_income + ctx.allowances

        net_pay = (
            total_earnings
            - nssf_result["employee"]
            - lst_amount
            - paye_result.amount
        )

        return {
            "basic_salary": ctx.gross_income,
            "allowances": ctx.allowances,
            "total_earnings": total_earnings,
            "employee_nssf": nssf_result["employee"],
            "employer_nssf": nssf_result["employer"],
            "lst": lst_amount,
            "paye": paye_result.amount,
            "net_pay": net_pay
        }


In [148]:
context = TaxContext(
    gross_income=Money("-100000"),
    allowances=Money("0"),
    residency_status="resident",
    effective_date=date(2026, 10, 1)
)

try:
    PAYE().calculate(context)
except InvalidIncomeError as e:
    print("Error caught:", e)

Error caught: Income cannot be negative


In [91]:
invoice = {
    "description": "Sale of goods",
    "amount": Money("100000"),
    "category": "standard-rated",
    "invoice_type": "sale"
}

In [150]:
purchase_invoice = {
    "description": "Office supplies",
    "amount": Money("50000"),
    "category": "standard-rated",
    "invoice_type": "purchase",
    "input_vat_eligible": True
}

ineligible_purchase = {
    "description": "Purchase not eligible for input VAT deduction",
    "amount": Money("30000"),
    "category": "standard-rated",
    "invoice_type": "purchase",
    "input_vat_eligible": False
}

In [93]:
zero_rated_invoice = {
    "description": "Zero-rated sale",
    "amount": Money("200000"),
    "category": "zero-rated",
    "invoice_type": "sale"
}

exempt_invoice = {
    "description": "Exempt sale",
    "amount": Money("150000"),
    "category": "exempt",
    "invoice_type": "sale"
}

invoices = [
    invoice,
    purchase_invoice,
    ineligible_purchase,
    zero_rated_invoice,
    exempt_invoice
]

In [151]:

class VAT:
    def __init__(self):
        self.standard_rate = Decimal("0.18")

    def calculate(self, invoices: list) -> dict:
        output_vat = Money("0")
        input_vat = Money("0")

        for invoice in invoices:
            amount = invoice["amount"].amount
            category = invoice["category"]
            invoice_type = invoice["invoice_type"]

            if category == "standard-rated":
                tax_amount = Money(amount * self.standard_rate)
            elif category in ("zero-rated", "exempt"):
                tax_amount = Money("0")
            else:
                raise ValueError("Invalid VAT category")

            if invoice_type == "sale":
                output_vat = output_vat + tax_amount
            elif invoice_type == "purchase":
                if invoice.get("input_vat_eligible", False):
                    input_vat = input_vat + tax_amount
            else:
                raise ValueError("Invalid invoice type")

        net_vat_payable = output_vat - input_vat

        return {
            "output_vat": output_vat,
            "input_vat": input_vat,
            "net_vat_payable": net_vat_payable
        }

In [152]:
vat = VAT()
vat_result = vat.calculate(invoices)

print("Output VAT:", vat_result["output_vat"])
print("Input VAT:", vat_result["input_vat"])
print("Net VAT payable:", vat_result["net_vat_payable"])

Output VAT: Money(18000.00)
Input VAT: Money(9000.00)
Net VAT payable: Money(9000.00)


In [153]:
for invoice in invoices:
    print(invoice["description"], "-", invoice["category"])
    
print("\nVAT results:")
print("Output VAT:", vat_result["output_vat"])
print("Input VAT:", vat_result["input_vat"])
print("Net VAT payable:", vat_result["net_vat_payable"])


Sale of goods - standard-rated
Office supplies - standard-rated
Purchase not eligible for input VAT deduction - standard-rated
Zero-rated sale - zero-rated
Exempt sale - exempt

VAT results:
Output VAT: Money(18000.00)
Input VAT: Money(9000.00)
Net VAT payable: Money(9000.00)


In [154]:
for i, invoice in enumerate(invoices, start=1):
    print(i, invoice["description"], invoice["invoice_type"],
          invoice["category"], invoice["amount"])

1 Sale of goods sale standard-rated Money(100000.00)
2 Office supplies purchase standard-rated Money(50000.00)
3 Purchase not eligible for input VAT deduction purchase standard-rated Money(30000.00)
4 Zero-rated sale sale zero-rated Money(200000.00)
5 Exempt sale sale exempt Money(150000.00)


In [155]:
class CorporateTax:
    def __init__(self):
        self.tax_rate = Decimal("0.30")

    def calculate(self, revenue: Money, allowable_deductions: Money) -> dict:
        if revenue.amount < 0:
            raise InvalidIncomeError("Revenue cannot be negative")

        if allowable_deductions.amount < 0:
            raise ValueError("Allowable deductions cannot be negative")

        taxable_profit = revenue - allowable_deductions

        # A company with no taxable profit should not have negative tax.
        if taxable_profit.amount < 0:
            taxable_profit = Money("0")

        tax_payable = Money(taxable_profit.amount * self.tax_rate)

        return {
            "taxable_profit": taxable_profit,
            "tax_payable": tax_payable
        }


In [160]:

corporate_tax = CorporateTax()

corporate_result = corporate_tax.calculate(
    revenue=Money("100000000"),
    allowable_deductions=Money("60000000")
)

print("Taxable profit:", corporate_result["taxable_profit"])
print("Corporate tax:", corporate_result["tax_payable"])


Taxable profit: Money(40000000.00)
Corporate tax: Money(12000000.00)


In [161]:
# Test 1: Profitable company
result_profit = corporate_tax.calculate(
    Money("100000000"),
    Money("60000000")
)

assert result_profit["taxable_profit"] == Money("40000000")
assert result_profit["tax_payable"] == Money("12000000")

# Test 2: Company with deductions exceeding revenue
result_loss = corporate_tax.calculate(
    Money("50000000"),
    Money("60000000")
)

assert result_loss["taxable_profit"] == Money("0")
assert result_loss["tax_payable"] == Money("0")

print("Corporate tax tests passed!")

Corporate tax tests passed!


In [162]:
class WithholdingTax:
    def __init__(self):
        self.tax_rate = Decimal("0.06")
    def calculate(self, payment: Money) -> dict:
        if payment.amount < 0:
            raise InvalidIncomeError("Payment cannot be negative")

        tax_withheld = Money(payment.amount * self.tax_rate)
        net_payment = payment - tax_withheld

        return {
            "tax_withheld": tax_withheld,
            "net_payment": net_payment
        }

In [163]:

withholding_tax = WithholdingTax()

wht_result = withholding_tax.calculate(
    Money("1000000")
)

print("Tax withheld:", wht_result["tax_withheld"])
print("Net payment:", wht_result["net_payment"])


Tax withheld: Money(60000.00)
Net payment: Money(940000.00)


In [187]:
test_incomes = [
    335000,
    335001,
    410000,
    485000,
    10000000,
    10000001
]

paye = PAYE()

for income in test_incomes:
    context = TaxContext(
        gross_income=Money(income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    result = paye.calculate(context)
    print(f"Income: UGX {income:,} | PAYE: {result.amount}")


Income: UGX 335,000 | PAYE: Money(0.00)
Income: UGX 335,001 | PAYE: Money(0.20)
Income: UGX 410,000 | PAYE: Money(15000.00)
Income: UGX 485,000 | PAYE: Money(33750.00)
Income: UGX 10,000,000 | PAYE: Money(2888250.00)
Income: UGX 10,000,001 | PAYE: Money(2888250.10)


In [188]:
expected_paye = [
    Money("0"),
    Money("0.20"),
    Money("15000"),
    Money("33750"),
    Money("2888250"),
    Money("2888250.10")
]

for income, expected in zip(test_incomes, expected_paye):
    context = TaxContext(
        gross_income=Money(income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    result = paye.calculate(context)

    assert result.amount == expected, (
        f"Failed for income UGX {income:,}: "
        f"expected {expected}, got {result.amount}"
    )

print("All six resident PAYE boundary tests passed!")


All six resident PAYE boundary tests passed!


In [166]:
@given(income=st.integers(min_value=0, max_value=100_000_000))
def test_paye_non_negative(income):
    context = TaxContext(
        gross_income=Money(income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    result = paye.calculate(context)

    assert result.amount.amount >= 0

In [167]:
test_paye_non_negative()
print("PAYE non-negative property test passed!")

PAYE non-negative property test passed!


In [168]:
@given(
    lower_income=st.integers(min_value=0, max_value=100_000_000),
    extra_income=st.integers(min_value=0, max_value=100_000_000)
)
def test_paye_monotonicity(lower_income, extra_income):
    higher_income = lower_income + extra_income

    lower_context = TaxContext(
        gross_income=Money(lower_income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    higher_context = TaxContext(
        gross_income=Money(higher_income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    lower_tax = paye.calculate(lower_context).amount
    higher_tax = paye.calculate(higher_context).amount

    assert higher_tax >= lower_tax


In [169]:
test_paye_monotonicity()
print("PAYE monotonicity property test passed!")

PAYE monotonicity property test passed!


In [170]:
@given(income=st.integers(min_value=0, max_value=100_000_000))
def test_paye_not_greater_than_income(income):
    context = TaxContext(
        gross_income=Money(income),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    tax = paye.calculate(context).amount

    assert tax.amount <= Money(income).amount

In [171]:
test_paye_not_greater_than_income()
print("PAYE not-greater-than-income property test passed!")

PAYE not-greater-than-income property test passed!


In [189]:
nonresident_test_cases = [
    (335000, "33500.00"),
    (335001, "33500.20"),
    (410000, "48500.00"),
    (410001, "48500.30"),
    (10000000, "2925500.00"),
    (10000001, "2925500.10")
]

for income, expected_tax in nonresident_test_cases:
    context = TaxContext(
        gross_income=Money(income),
        allowances=Money("0"),
        residency_status="non-resident",
        effective_date=date(2026, 10, 1)
    )

    result = paye.calculate(context)

    assert result.amount.amount == Decimal(expected_tax), (
        f"Income {income}: expected {expected_tax}, "
        f"got {result.amount.amount}"
    )

print("All nonresident PAYE boundary tests passed!")

All nonresident PAYE boundary tests passed!


In [173]:
# Test 1: Negative income
try:
    context = TaxContext(
        gross_income=Money(-1000),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )
    paye.calculate(context)
    print("FAILED: Negative income was accepted")
except InvalidIncomeError:
    print("PASSED: Negative income was rejected")


# Test 2: Invalid residency status
try:
    context = TaxContext(
        gross_income=Money(500000),
        allowances=Money("0"),
        residency_status="visitor",
        effective_date=date(2026, 10, 1)
    )
    paye.calculate(context)
    print("FAILED: Invalid residency status was accepted")
except ValueError:
    print("PASSED: Invalid residency status was rejected")


# Test 3: Unsupported effective date
try:
    context = TaxContext(
        gross_income=Money(500000),
        allowances=Money("0"),
        residency_status="resident",
        effective_date=date(2026, 6, 30)
    )
    paye.calculate(context)
    print("FAILED: Unsupported date was accepted")
except NoScheduleForDateError:
    print("PASSED: Unsupported effective date was rejected")

PASSED: Negative income was rejected
PASSED: Invalid residency status was rejected
PASSED: Unsupported effective date was rejected


In [174]:
# Test NSSF contributions on UGX 500,000
nssf = NSSF()
result = nssf.calculate(Money(500000))

assert result["employee"].amount == Decimal("25000.00")
assert result["employer"].amount == Decimal("50000.00")

print("PASSED: Employee NSSF is 5%")
print("PASSED: Employer NSSF is 10%")


# Test rejection of negative income
try:
    nssf.calculate(Money(-1000))
    print("FAILED: Negative income was accepted")
except InvalidIncomeError:
    print("PASSED: Negative income was rejected")

PASSED: Employee NSSF is 5%
PASSED: Employer NSSF is 10%
PASSED: Negative income was rejected


In [175]:
vat = VAT()
vat_result = vat.calculate(invoices)

assert vat_result["output_vat"].amount == Decimal("18000.00")
assert vat_result["input_vat"].amount == Decimal("9000.00")
assert vat_result["net_vat_payable"].amount == Decimal("9000.00")

print("PASSED: Output VAT is correct")
print("PASSED: Eligible input VAT is correct")
print("PASSED: Net VAT payable is correct")

PASSED: Output VAT is correct
PASSED: Eligible input VAT is correct
PASSED: Net VAT payable is correct


In [176]:
# Test 1: Invalid VAT category
invalid_category_invoice = {
    "description": "Invalid category test",
    "amount": Money("100000"),
    "category": "reduced-rated",
    "invoice_type": "sale"
}

try:
    vat.calculate([invalid_category_invoice])
    print("FAILED: Invalid VAT category was accepted")
except ValueError:
    print("PASSED: Invalid VAT category was rejected")


# Test 2: Invalid invoice type
invalid_type_invoice = {
    "description": "Invalid invoice type test",
    "amount": Money("100000"),
    "category": "standard-rated",
    "invoice_type": "refund"
}

try:
    vat.calculate([invalid_type_invoice])
    print("FAILED: Invalid invoice type was accepted")
except ValueError:
    print("PASSED: Invalid invoice type was rejected")

PASSED: Invalid VAT category was rejected
PASSED: Invalid invoice type was rejected


In [177]:
corporate_tax = CorporateTax()

# Test 1: Profitable company
result = corporate_tax.calculate(
    Money(100_000_000),
    Money(60_000_000)
)

assert result["taxable_profit"].amount == Decimal("40000000.00")
assert result["tax_payable"].amount == Decimal("12000000.00")

print("PASSED: Corporate tax on taxable profit is correct")


# Test 2: Deductions exceed revenue
result = corporate_tax.calculate(
    Money(50_000_000),
    Money(60_000_000)
)

assert result["taxable_profit"].amount == Decimal("0.00")
assert result["tax_payable"].amount == Decimal("0.00")

print("PASSED: Tax is zero when taxable profit is negative")


# Test 3: Negative revenue
try:
    corporate_tax.calculate(Money(-1000), Money(0))
    print("FAILED: Negative revenue was accepted")
except InvalidIncomeError:
    print("PASSED: Negative revenue was rejected")


# Test 4: Negative deductions
try:
    corporate_tax.calculate(Money(100000), Money(-1000))
    print("FAILED: Negative deductions were accepted")
except ValueError:
    print("PASSED: Negative deductions were rejected")

PASSED: Corporate tax on taxable profit is correct
PASSED: Tax is zero when taxable profit is negative
PASSED: Negative revenue was rejected
PASSED: Negative deductions were rejected


In [178]:
withholding_tax = WithholdingTax()

# Test 1: Calculate withholding tax
result = withholding_tax.calculate(Money(1_000_000))

assert result["tax_withheld"].amount == Decimal("60000.00")
assert result["net_payment"].amount == Decimal("940000.00")

print("PASSED: Withholding tax calculation is correct")
print("PASSED: Net payment after withholding is correct")


# Test 2: Zero payment
result = withholding_tax.calculate(Money(0))

assert result["tax_withheld"].amount == Decimal("0.00")
assert result["net_payment"].amount == Decimal("0.00")

print("PASSED: Zero payment produces zero withholding tax")


# Test 3: Negative payment
try:
    withholding_tax.calculate(Money(-1000))
    print("FAILED: Negative payment was accepted")
except InvalidIncomeError:
    print("PASSED: Negative payment was rejected")

PASSED: Withholding tax calculation is correct
PASSED: Net payment after withholding is correct
PASSED: Zero payment produces zero withholding tax
PASSED: Negative payment was rejected


In [179]:
paye = PAYE()
nssf = NSSF()
lst = LST(amount=5000)

payroll = Payroll(paye, nssf, lst)

context = TaxContext(
    gross_income=Money(500000),
    allowances=Money(0),
    residency_status="resident",
    effective_date=date(2026, 10, 1)
)

result = payroll.calculate(context)

assert result["employee_nssf"].amount == Decimal("25000.00")
assert result["employer_nssf"].amount == Decimal("50000.00")
assert result["lst"].amount == Decimal("5000.00")
assert result["paye"].amount == Decimal("38250.00")
assert result["net_pay"].amount == Decimal("431750.00")

print("PASSED: Employee NSSF is correct")
print("PASSED: Employer NSSF is correct")
print("PASSED: LST is correct")
print("PASSED: PAYE is correct")
print("PASSED: Net pay is correct")

PASSED: Employee NSSF is correct
PASSED: Employer NSSF is correct
PASSED: LST is correct
PASSED: PAYE is correct
PASSED: Net pay is correct


In [55]:
try:
    invalid_context = TaxContext(
        gross_income=Money(-1000),
        allowances=Money(0),
        residency_status="resident",
        effective_date=date(2026, 10, 1)
    )

    payroll.calculate(invalid_context)

    print("FAILED: Payroll accepted negative gross income")

except InvalidIncomeError:
    print("PASSED: Payroll rejected negative gross income")

PASSED: Payroll rejected negative gross income


In [180]:
context_without_allowances = TaxContext(
    gross_income=Money(500000),
    allowances=Money(0),
    residency_status="resident",
    effective_date=date(2026, 10, 1)
)

context_with_allowances = TaxContext(
    gross_income=Money(500000),
    allowances=Money(100000),
    residency_status="resident",
    effective_date=date(2026, 10, 1)
)

tax_without_allowances = paye.calculate(context_without_allowances)
tax_with_allowances = paye.calculate(context_with_allowances)

print("Tax without allowances:", tax_without_allowances.amount)
print("Tax with allowances:", tax_with_allowances.amount)

Tax without allowances: Money(38250.00)
Tax with allowances: Money(68250.00)


In [182]:

class Payroll:
    def __init__(self, paye, nssf, lst):
        self.paye = paye
        self.nssf = nssf
        self.lst = lst

    def calculate(self, ctx: TaxContext) -> dict:
        # Calculate employee and employer NSSF contributions.
        nssf_result = self.nssf.calculate(ctx.gross_income)

        # Calculate configurable Local Service Tax.
        lst_amount = self.lst.calculate()

        # Calculate PAYE using gross income plus taxable allowances.
        paye_result = self.paye.calculate(ctx)

        # Include allowances when calculating total earnings.
        total_earnings = ctx.gross_income + ctx.allowances

        # Deduct employee-side contributions and taxes.
        net_pay = (
            total_earnings
            - nssf_result["employee"]
            - lst_amount
            - paye_result.amount
        )

        return {
            "basic_salary": ctx.gross_income,
            "allowances": ctx.allowances,
            "total_earnings": total_earnings,
            "employee_nssf": nssf_result["employee"],
            "employer_nssf": nssf_result["employer"],
            "lst": lst_amount,
            "paye": paye_result.amount,
            "net_pay": net_pay
        }


In [186]:
payroll = Payroll(
    paye=PAYE(),
    nssf=NSSF(),
    lst=LST()
)

context = TaxContext(
    gross_income=Money(500000),
    allowances=Money(100000),
    residency_status="resident",
    effective_date=date(2026, 10, 1)
)

result = payroll.calculate(context)

for key, value in result.items():
    print(f"{key}: {value}")

assert result["total_earnings"].amount == Decimal("600000.00")
assert result["paye"].amount == Decimal("68250.00")
assert result["net_pay"].amount == Decimal("506750.00")

print("PASSED: Payroll includes allowances correctly.")


basic_salary: Money(500000.00)
allowances: Money(100000.00)
total_earnings: Money(600000.00)
employee_nssf: Money(25000.00)
employer_nssf: Money(50000.00)
lst: Money(0.00)
paye: Money(68250.00)
net_pay: Money(506750.00)
PASSED: Payroll includes allowances correctly.


#Why is `float` unsafe for monetary calculations?

Python's `float` data type represents numbers using binary floating-point arithmetic. Some decimal fractions cannot be represented exactly in binary, which can introduce small rounding errors. For example, `0.1 + 0.2` produces approximately `0.30000000000000004` rather than exactly `0.3`. In financial calculations, repeated small errors can affect tax amounts, deductions, and balances.

#Where is inheritance justified or should it be avoided?

Inheritance is justified when different classes represent related types that share a common interface and can be used interchangeably. In this project, the abstract `Tax` class defines the `calculate(self, ctx: TaxContext) -> TaxResult` contract. Tax implementations that follow this contract can inherit from `Tax`, ensuring that they provide their own calculation method. This supports polymorphism and makes it easier to extend the system with additional tax types.

Inheritance should be avoided when classes do not share the same behaviour or interface. For example, the current NSSF calculation returns both employee and employer contributions, while VAT calculations work with invoices and categories. Forcing these classes into the same inheritance structure could make the design less clear. The `Payroll` class instead uses composition: it receives PAYE, NSSF, and LST objects and combines their results. This is appropriate because payroll coordinates several distinct calculations rather than being a tax calculation of its own.

The design should therefore use inheritance for genuine shared contracts and composition where one component needs to coordinate other components.